In [44]:
import json
import pandas as pd
from pathlib import Path

In [45]:
edition_profile = json.load(open("../json-copy/phi0830.phi001.new_edition_profile.json"))

file_path = Path("./CSV/calpsicapp.csv")

df = pd.read_csv(file_path)

In [46]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4715 entries, 0 to 4714
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   app_id               4715 non-null   str    
 1   id                   4715 non-null   str    
 2   cts_urn              4715 non-null   str    
 3   poem_number          4715 non-null   int64  
 4   line_number          4715 non-null   int64  
 5   claim_type           4715 non-null   str    
 6   reading_type         4715 non-null   str    
 7   reading              4560 non-null   str    
 8   supporting_evidence  3523 non-null   str    
 9   note                 405 non-null    str    
 10  reporting_rank       3102 non-null   float64
dtypes: float64(1), int64(2), str(8)
memory usage: 405.3 KB


In [47]:
unique_app_ids = df['app_id'].unique()

In [48]:
# Get unique sigla
def get_unique_sigla(df):
    supporting_evidence = df['supporting_evidence'].unique()
    sigla_list = []
    for item in supporting_evidence:
        if isinstance(item, str):
            sigla = item.split()
            for s in sigla:
                if s not in sigla_list:
                    sigla_list.append(s)
    return sigla_list

In [49]:
edition_sigla = get_unique_sigla(df)

In [50]:
source_concordance = {
    "Heins.": "Heins.",
    "Romana": "r",
    "Barth1613": "Barth 1613",
    "Baehr1870": "Baehr. 1870",
    "Burm1731": "Burm. 1731",
    "Wilamowitz": "Wilamowitz-Moellendorff 1884",
    "Friesemann": "Friesemann 1786",
    "Dempster": "Dempster 1723",
    "Hermann": "Hermann 1849",
    "Haupt1854": "Haupt 1854",
    "Leo": "Leo 1885",
    "Barth1624": "Barth 1624",
    "Mueller": "Müller 1885",
    "Postgate": "Postgate 1901",
    "Nodell": "Nodell 1787",
    "Ellis": "Ellis 1886",
    "Gronovius1755": "Gronovius 1755",
    "Haupt1874": "Haupt 1874",
    "Heraldus": "Heraldus",
    "Modius": "Modius 1854",
    "Gebhard": "Gebhard 1615",
    "ExcPar": "Exc. Par.",
    "deRooy": "de Rooy 1771",
    "Heins1665": "Heins. ad Claud.",
    "Voss": "Voss 1797",
    "Gebauer": "Gebauer 1860",
    "Baehrens1872": "Baehr. 1872",
    "CSchenkl": "C. Schenkl",
    "Vigilius": "Barthius",
    "Scriverius": "Scriver.",
    "Fabricius": "Fabricius 1774",
    "ExcBon": "Exc. Bon.",
    "Salmasius1656": "Salmasius 1656",
    "ExcParPrior": "Exc. Par. Prior",
    "Heins1661": "Heins. 1661",
    "Buecheler1871": "Buecheler 1871",
    "Bursian": "Bursian",
    "Hoeufft": "Hoeufft 1783",
    "Wakefield": "Wakefield 1789",
    "DOrville": "D’Orville",
    "Housman1902": "Housman 1902",
    "ExcParAlter": "Exc. Par. Alter",
    "Lucas": "Lucan 1901",
    "Bergk": "Bergk 1865",
    "codVind": "cod. Vindob.",
    "Brodaeus": "Brodaeus",
    "Erasmus": "Erasmus 1703",
    "Kannegieter": "Kannegieter 1731",
    "Tortell.": "Tortell.",
    "Chytil": "Chytil 1893–94",
    "Iustus": "Iustus 1765",
    "Schraeder": "Schraeder",
    "Carrio": "Carrion 1583",
    "Salmasius1620": "Salmasius 1620",
    "Spanheim": "Spanheim 1717",
    "Bartholinus": "Bartholinus 1647",
}

In [51]:
# Functions
def process_value(value):
    # If the value is a float and is NaN, return None. Otherwise, return the value.
    if isinstance(value, float) and pd.isna(value):
        return None
    elif isinstance(value, float) and not pd.isna(value):
        return int(value)
    elif pd.isna(value):
        return None
    else:
        return value

def process_evidence(evidence_str):
    if pd.isna(evidence_str):
        return []
    else:
        evidence_list = [e.strip() for e in evidence_str.split()]
        evidence = []
        not_found = []
        sources = (
            edition_profile["sigla"]
            + edition_profile["people"]
            + edition_profile["secondary_sources"]
        )
        for e in evidence_list:
            if len(e) > 2:
                entry = next(
                    (
                        item
                        for item in sources
                        if item.get("label") == e or item.get("label") == e
                    ),
                    None,
                )
                if entry:
                    item = {}
                    item["id"] = entry["uuid"]
                    item["label"] = e
                    evidence.append(item)
                else:
                    continue
            elif len(e) == 2:
                e_list = [i for i in e]
                label = e_list[0]
                hand = e_list[1]
                entry = next(
                    (
                        item
                        for item in sources
                        if item.get("label") == label
                        or item.get("label") == label
                    ),
                    None,
                )
                if entry:
                    item = {}
                    item["id"] = entry["uuid"]
                    item["label"] = label
                    item["hand"] = hand
                    evidence.append(item)
                else:
                    continue
            elif len(e) == 1:
                entry = next(
                    (
                        item
                        for item in sources
                        if item.get("label") == e
                    ),
                    None,
                )
                if entry:
                    item = {}
                    item["id"] = entry["uuid"]
                    item["label"] = e
                    evidence.append(item)
                else:
                    continue
            else:
                not_found.append(e)
    for e in set(not_found):
        print(f"Not found: {e}")                
    return evidence

def create_json_structure(df):
    df.fillna(None, inplace=True)
    data = []
    for row in df.itertuples():
        entry = {}
        entry['type'] = "apparatus_claim"
        entry['id'] = process_value(row[2])
        entry['apparatus_entry_id'] = process_value(row[1])
        entry['target'] = process_value(row[3])
        entry['claim_type'] = process_value(row[6])
        entry['reading_type'] = process_value(row[7])
        entry['reading'] = process_value(row[8])
        entry['supporting_evidence'] = process_evidence(row[9])       
        entry["reporting_rank"] = process_value(row[11])
        entry['editorial_comments'] = process_value(row[10])
        entry["confidence"] = {"interpretation": None, "reasons": []}
        entry["evaluation"] = {
            "target_correct": None,
            "reading_correct": None,
            "witnesses_correct": None,
            "interpretation_correct": None,
            "reviewer": None,
            "review_status": None,
        }
        data.append(entry)

    return data

In [52]:
my_json = create_json_structure(df)

In [53]:
json_output_path = Path("calpsicapp3.json")
with open(json_output_path, "w", encoding="utf-8") as f:
    json.dump([i for i in my_json], f, ensure_ascii=False, indent=4)

In [54]:
print([json.dumps(i, ensure_ascii=False, indent=4) for i in my_json])

['{\n    "type": "apparatus_claim",\n    "id": "2941e83d-e1f3-4d41-863b-5f86a89c6890",\n    "apparatus_entry_id": "b7d5b65a-ad2f-495d-aed3-cc6bbddcf09c",\n    "target": "urn:cts:latinLit:phi0830.phi001.dll:1.1@C.",\n    "claim_type": "reading",\n    "reading_type": "lemma",\n    "reading": "C.",\n    "supporting_evidence": [\n        {\n            "id": "urn:uuid:5c542460-a47b-4ec5-8cb5-61c2a1b3bd39",\n            "label": "G"\n        },\n        {\n            "id": "urn:uuid:813d3a5f-1fb8-4905-b10a-8dca34d64509",\n            "label": "P"\n        },\n        {\n            "id": "urn:uuid:95b19ca1-7f6a-459b-a461-9697623d2284",\n            "label": "A"\n        },\n        {\n            "id": "urn:uuid:ea3aba4e-5a9d-4cd1-a782-724d50639fe5",\n            "label": "φ"\n        },\n        {\n            "id": "urn:uuid:47e5f301-0070-4036-99c6-11bdd756559a",\n            "label": "Ulit."\n        },\n        {\n            "id": "urn:uuid:8e9640ff-7b06-41ce-87a9-3e66001ca8cc",\n    

In [41]:
evidence_str = "G P A φ Ulit. Wernsd. N2 λ"

In [42]:
evidence_list = [e.strip() for e in evidence_str.split()]
evidence = []
not_found = []
two_char_witnesses = []
sources = edition_profile['sigla'] + edition_profile['people'] + edition_profile['secondary_sources']
for e in evidence_list:
    # If e is a key in the edition_concordance, assign the value to e
    if e in source_concordance.keys():
        e = source_concordance[e]
        not_found.append(e)
        
        if len(e) > 2:
            entry = next((item for item in sources if item.get('siglum') == e or item.get('abbreviation') == e), None)
            if entry:
                item = {}
                item['id'] = entry['uuid']
                item['label'] = e
                evidence.append(item)
            else:
                not_found.append(e)
        elif len(e) == 2:
            e_list = [i for i in e]
            siglum = e_list[0]
            hand = e_list[1]
            entry = next((item for item in sources if item.get('siglum') == siglum or item.get('abbreviation') == siglum), None)
            if entry:
                item = {}
                item['id'] = entry['uuid']
                item['label'] = siglum
                item['hand'] = hand
                evidence.append(item)
            else:
                not_found.append(siglum)
        elif len(e) == 1:
            entry = next((item for item in sources if item.get('siglum') == e or item.get('abbreviation') == e), None)
            if entry:
                item = {}
                item['id'] = entry['uuid']
                item['label'] = e
                evidence.append(item)
            else:
                not_found.append(e)

#             # if e not in (object.get('siglum'), object.get('abbreviation')):
#             #         not_found.append(e)
#             #     else:
#             #         item = {}
#             #         item['id'] = object['uuid']
#             #         item['label'] = e
#             #         evidence.append(item)
#             # elif len(e) == 2:
#             #     e_list = [i for i in e]
#             #     print(e_list)
#             #     # siglum = e_list[0]
#             #     # hand = e_list[1]
#             #     # if siglum not in (object.get('siglum'), object.get('abbreviation')):
#             #     #     not_found.append(siglum)
#             #     # else:
#             #     #     item = {}
#             #     #     item['id'] = object['uuid']
#             #     #     item['label'] = e
#             #     #     item['hand'] = hand
#             #     #     evidence.append(item)
#             # # elif len(e) == 1:
#             # #     if e not in (object.get('siglum'), object.get('abbreviation')):
#             # #         print(f"Witness '{e}' not found.")
#             # #     else:
#             # #         item = {}
#             # #         item['id'] = object['uuid']
#             # #         item['label'] = e
#             # #         evidence.append(item)
#             # # else:
#             # #     print(f"Witness '{e}' not found.'")

In [43]:
for e in set(not_found):
    print(e)

for e in set(two_char_witnesses):
    print(e)